# 第 07 章 已知标记基因与可视化

## 学习目标

学习用多种图形展示已知标记基因，建立注释所需的证据。

## 为什么做与怎样做

整理粗粒度与细粒度标记集合，检查基因是否存在；比较表达强度、表达比例和细胞分布。z-score 仅用于需要缩放的绘图副本。

前置章节：06。运行前请完成项目环境准备。

本章在项目副本运行。遇到等待材料/确认，请按根目录 **99_运行与AI协作指南.md** 查看当前结果、与用户讨论并续跑；不能跳过待办直接进入下游。


In [ ]:
# 功能说明：查找公共环境和当前项目，读取有效上游检查点；模板副本之间不共享分析状态。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
from __future__ import annotations
from pathlib import Path
import os
import sys
ROOT = Path(os.environ.get("SC_COURSE_ROOT", Path.cwd())).resolve()
while not (ROOT / "config/course.json").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config/course.json").is_file():
    raise RuntimeError("请从课程目录或其 notebooks/exercises 目录运行。")
os.environ["CELLTYPIST_FOLDER"] = str(ROOT / ".runtime/celltypist")
os.environ["MPLCONFIGDIR"] = str(ROOT / ".runtime/matplotlib")
sys.path.insert(0, str(ROOT / "tools"))
from course_runtime import start_chapter, marker_sets, scaled_view
from course_projects import resolve_project, input_path, read_sample
from course_resources import qc_gene_sets, marker_resources, resolution_preview
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation
INSTALL_ROOT = ROOT
ROOT = resolve_project()
ctx = start_chapter("07")
adata = ctx.load_input()



## 07.1 以已知的标记基因注释

In [ ]:
# 功能说明：更新图形输出目录。
# 运行目的：将基于标记基因注释的图片保存到特定子文件夹。
# 详细代码解析：
# 1. `sc.settings.figdir = ...`
#    - 设置新的输出路径。

# 设置输出目录
sc.settings.figdir = ctx.figures

## 07.2 标记基因整理
我们为本数据集中主要细胞类型定义一组标记基因。该集合改编自 [Single Cell Best Practices 的注释章节](https://www.sc-best-practices.org/cellular_structure/annotation.html)。如需更详细的概览与最佳实践，请参考该章节。

In [ ]:
# 功能说明：定义主要细胞类型的标记基因字典。
# 运行目的：用于 dotplot/UMAP 等可视化及手动注释。
# 变量/函数/参数解析：
# - markers(dict[str, list[str]])：键为细胞类型名称，值为该类型的候选标记基因列表。
# 注意：其中少数基因为负标记（不表达），已在注释中注明。
markers_1 = {
    "CD14+ Mono": ["FCN1", "CD14"],
    "CD16+ Mono": ["TCF7L2", "FCGR3A", "LYN"],
    # 注意：DMXL2 应为负标记
    "cDC2": ["CST3", "COTL1", "LYZ", "DMXL2", "CLEC10A", "FCER1A"],
    "Erythroblast": ["MKI67", "HBA1", "HBB"],
    # 注意：HBM 与 GYPA 为负标记
    "Proerythroblast": ["CDK6", "SYNGR1", "HBM", "GYPA"],
    "NK": ["GNLY", "NKG7", "CD247", "FCER1G", "TYROBP", "KLRG1", "FCGR3A"],
    "ILC": ["ID2", "PLCG2", "GNLY", "SYNE1"],
    "Naive CD20+ B": ["MS4A1", "IL4R", "IGHD", "FCRL1", "IGHM"],
    # 注意：IGHD 与 IGHM 为负标记
    "B cells": [
        "MS4A1",
        "ITGB1",
        "COL4A4",
        "PRDM1",
        "IRF4",
        "PAX5",
        "BCL11A",
        "BLK",
        "IGHD",
        "IGHM",
    ],
    "Plasma cells": ["MZB1", "HSP90B1", "FNDC3B", "PRDM1", "IGKC", "JCHAIN"],
    # 注意：PAX5 为负标记
    "Plasmablast": ["XBP1", "PRDM1", "PAX5"],
    "CD4+ T": ["CD4", "IL7R", "TRBC2"],
    "CD8+ T": ["CD8A", "CD8B", "GZMK", "GZMA", "CCL5", "GZMB", "GZMH", "GZMA"],
    "T naive": ["LEF1", "CCR7", "TCF7"],
    "pDC": ["GZMB", "IL3RA", "COBLL1", "TCF4"],
    "platelet":  ["ITGA2B", "PF4", "TUBB1", "PPBP", ]
}
markers_1 = marker_resources(ctx)["fine"]
missing_markers = {ct: [g for g in gs if g not in adata.var_names] for ct, gs in markers_1.items()}


In [ ]:
# 仅保留在我们数据中检测到的标记。我们将循环遍历所有细胞类型，并仅保留我们在 adata 对象中找到的基因作为该细胞类型的标记。这将防止我们在开始绘图时出现错误。
# 创建一个空字典，用于存储每个细胞类型在数据集中实际存在的标记基因
# 格式：{细胞类型1: [存在的基因1, 存在的基因2, ...], 细胞类型2: [...]}
markers_in_data = {}
# 遍历原始标记基因字典中的每一个键值对
# ct: 细胞类型名称 (cell type)
# markers: 该细胞类型对应的标记基因列表
for ct, markers in markers_1.items():   
    # 为当前细胞类型创建一个空列表，用于存储在数据集中实际找到的标记基因
    markers_found = []    
    # 遍历当前细胞类型的所有标记基因
    for marker in markers:
        # 检查当前标记基因是否存在于adata的基因索引中
        # adata.var.index 通常包含数据集中所有基因的名称
        if marker in adata.var.index:
            # 如果基因存在，将其添加到找到的基因列表中
            markers_found.append(marker)    
    # 将当前细胞类型及其在数据集中实际存在的标记基因列表存入结果字典
    markers_in_data[ct] = markers_found
markers_1 = markers_in_data
# 代码执行完毕后：
# markers_in_data 字典将只包含那些在数据集中实际存在的标记基因
# 如果某个细胞类型的所有标记基因都不在数据集中，对应的值将为空列表

In [ ]:
# 功能说明：查看过滤后的标记基因字典。
# 运行目的：确认哪些标记基因在当前数据集中实际存在。
# 详细代码解析：
# 1. `markers_1`
#    - 打印字典内容。

markers_1


| 细胞大类 (Major Class) | 细胞类型 (Cell Type) | 主要谱系 | 参考标记（需组合验证） | 主要生物学功能 | 主要关联/分化关系 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **髓系白细胞 (Myeloid Leukocytes)** | **CD14+ 单核细胞 (CD14+ Monocyte)** | 髓系 | **FCN1, CD14** | 炎症单核，强吞噬与抗原呈递能力 | 可分化至巨噬细胞、DC或CD16+单核 |
| **髓系白细胞 (Myeloid Leukocytes)** | **CD16+ 单核细胞 (CD16+ Monocyte)** | 髓系 | **TCF7L2, FCGR3A, LYN** | 巡逻单核，免疫监视，抗病毒应答强 | 与CD14+ Mono构成功能连续体 |
| **髓系白细胞 (Myeloid Leukocytes)** | **经典树突状细胞2型 (cDC2)** | 髓系 | **CST3, COTL1, LYZ, CLEC10A, FCER1A**（另查看 DMXL2） | 经典树突状细胞2型，激活CD4+ T细胞 | 起源于髓系前体 |
| **髓系白细胞 (Myeloid Leukocytes)** | **浆细胞样树突状细胞 (pDC)** | 髓系（近淋）| **GZMB, IL3RA, COBLL1, TCF4** | 浆细胞样树突状细胞，大量产生I型干扰素 | 专职抗病毒免疫 |
| **红细胞系 (Erythroid)** | **早幼红细胞 (Proerythroblast)** | 红系前体 | **CDK6, SYNGR1**（负标记：HBM, GYPA） | 早期红细胞前体，高增殖潜力 | 分化为 **Erythroblast** |
| **红细胞系 (Erythroid)** | **幼红细胞 (Erythroblast)** | 红系前体 | **HBA1, HBB, MKI67** | 晚期红细胞前体，合成血红蛋白 | 来自 **Proerythroblast** |
| **B细胞及终末分化群体 (B Cells and Terminal Descendants)** | **初始CD20+ B细胞 (Naive CD20+ B cell)** | 淋巴系 (B) | **MS4A1, IL4R, FCRL1**（负标记：IGHD, IGHM） | 初始B细胞，未接触抗原 | 为B细胞激活的起点 |
| **B细胞及终末分化群体 (B Cells and Terminal Descendants)** | **B细胞 (B cells (general))** | 淋巴系 (B) | **MS4A1, ITGB1, COL4A4, PRDM1, IRF4, PAX5, BCL11A, BLK**（负标记：IGHD, IGHM） | 广义B细胞（含初始与记忆） | 可分化为 **Plasmablast** |
| **B细胞及终末分化群体 (B Cells and Terminal Descendants)** | **浆母细胞 (Plasmablast)** | 淋巴系 (B) | **XBP1, PRDM1**（负标记：PAX5） | 增殖中的抗体分泌细胞，中间状态 | 分化为 **Plasma cells** |
| **B细胞及终末分化群体 (B Cells and Terminal Descendants)** | **浆细胞 (Plasma cells)** | 淋巴系 (B) | **MZB1, HSP90B1, FNDC3B, PRDM1, IGKC, JCHAIN** | 终末浆细胞，专职分泌抗体 | 来自 **Plasmablast** |
| **T细胞、NK及固有淋巴细胞 (T Cells, NK & ILCs)** | **初始T细胞 (T naive cell)** | 淋巴系 (T) | **LEF1, CCR7, TCF7** | 初始T细胞，处于静息状态 | 分化为 **CD4+ T** 或 **CD8+ T** |
| **T细胞、NK及固有淋巴细胞 (T Cells, NK & ILCs)** | **CD4+ T细胞 (CD4+ T cell)** | 淋巴系 (T) | **CD4, IL7R, TRBC2** | 辅助T细胞，协调免疫应答 | 来自 **T naive** |
| **T细胞、NK及固有淋巴细胞 (T Cells, NK & ILCs)** | **CD8+ T细胞 (CD8+ T cell)** | 淋巴系 (T) | **CD8A, CD8B, GZMK, GZMA, CCL5, GZMB, GZMH** | 细胞毒性T细胞，直接杀伤靶细胞 | 来自 **T naive**；功能与NK相似 |
| **T细胞、NK及固有淋巴细胞 (T Cells, NK & ILCs)** | **自然杀伤细胞 (NK cell)** | 淋巴系 (ILC) | **GNLY, NKG7, CD247, FCER1G, TYROBP, KLRG1, FCGR3A** | 自然杀伤细胞，固有免疫杀伤 | 与CD8+ T功能互补，无抗原特异性 |
| **T细胞、NK及固有淋巴细胞 (T Cells, NK & ILCs)** | **固有淋巴细胞 (ILC)** | 淋巴系 (ILC) | **ID2, PLCG2, GNLY, SYNE1** | 固有淋巴细胞，组织驻留免疫哨兵 | 功能类似T细胞，但属固有免疫 |
分类说明：

髓系白细胞：在本课程中用于归纳单核细胞和树突状细胞，具体发育来源需结合细胞亚型判断，它们主要负责先天免疫、抗原呈递和免疫调节。

红细胞系：专门归集红细胞生成过程中的前体细胞，与免疫细胞功能路径不同。

B细胞及终末分化群体：将B细胞谱系从初始阶段到终末的抗体分泌细胞（浆细胞）归为一类，体现了该谱系的完整分化轨迹。

T细胞、NK及固有淋巴细胞：将具有细胞毒性、辅助性和固有免疫功能的淋巴细胞归为一类，它们在细胞介导的免疫中扮演核心角色，功能上互补且常被同时分析。

In [ ]:
# ## 1. 髓系白细胞 (Myeloid Leukocytes)
# - **包含细胞类型**：CD14+ Mono、CD16+ Mono、cDC2、pDC
# - **标记基因集合**：`['FCN1', 'CD14', 'TCF7L2', 'FCGR3A', 'LYN', 'CST3', 'COTL1', 'LYZ', 'DMXL2', 'CLEC10A', 'FCER1A', 'GZMB', 'IL3RA', 'COBLL1', 'TCF4']`
# ## 2. 红细胞系 (Erythroid)
# - **包含细胞类型**：Proerythroblast、Erythroblast
# - **标记基因集合**：`['CDK6', 'SYNGR1', 'HBM', 'GYPA', 'MKI67', 'HBA1', 'HBB']`
# ## 3. B细胞及终末分化群体 (B Cells and Terminal Descendants)
# - **包含细胞类型**：Naive CD20+ B、B cells、Plasmablast、Plasma cells
# - **标记基因集合**：`['MS4A1', 'IL4R', 'IGHD', 'FCRL1', 'IGHM', 'ITGB1', 'COL4A4', 'PRDM1', 'IRF4', 'PAX5', 'BCL11A', 'BLK', 'XBP1', 'MZB1', 'HSP90B1', 'FNDC3B', 'IGKC', 'JCHAIN']`
# ## 4. T细胞、NK及固有淋巴细胞 (T Cells, NK & ILCs)
# - **包含细胞类型**：T naive、CD4+ T、CD8+ T、NK、ILC
# - **标记基因集合**：`['LEF1', 'CCR7', 'TCF7', 'CD4', 'IL7R', 'TRBC2', 'CD8A', 'CD8B', 'GZMK', 'GZMA', 'CCL5', 'GZMB', 'GZMH', 'GNLY', 'NKG7', 'FCER1G', 'TYROBP', 'KLRG1', 'FCGR3A', 'ID2', 'PLCG2', 'SYNE1']`

markers_2={
            'Myeloid_Leukocytes': [
                'FCN1', 'CD14', 'TCF7L2', 'FCGR3A', 'LYN', 'CST3', 'COTL1', 
                'LYZ', 'DMXL2', 'CLEC10A', 'FCER1A', 'GZMB', 'IL3RA', 'COBLL1', 'TCF4'
            ],
            'Erythroid': [
                'CDK6', 'SYNGR1', 'HBM', 'GYPA', 'MKI67', 'HBA1', 'HBB'
            ],
            'B_Cells': [
                'MS4A1', 'IL4R', 'IGHD', 'FCRL1', 'IGHM', 'ITGB1', 'COL4A4', 
                'PRDM1', 'IRF4', 'PAX5', 'BCL11A', 'BLK', 'XBP1', 'MZB1', 
                'HSP90B1', 'FNDC3B', 'IGKC', 'JCHAIN'
            ],
            'T_NK_ILCs': [
                'LEF1', 'CCR7', 'TCF7', 'CD4', 'IL7R', 'TRBC2', 'CD8A', 
                'CD8B', 'GZMK', 'GZMA', 'CCL5', 'GZMB', 'GZMH', 'GNLY', 
                'NKG7', 'FCER1G', 'TYROBP', 'KLRG1', 'FCGR3A', 'ID2', 'PLCG2', 'SYNE1'
            ]
        }
markers_2 = marker_resources(ctx)["broad"]
markers_2 = {ct: [g for g in gs if g in adata.var_names] for ct, gs in markers_2.items()}


In [ ]:
# 变量/函数/参数解析：
# - markers_2：当前项目实际匹配的粗粒度 marker，各分辨率使用同一集合便于比较。
# - coarse_key/fine_key：由用户选择的真实 obs 聚类列名，不再固定为某两个数字。
# - resolution_preview：保存相同尺度的表达表/比例表和点图，再依次确认粗/细粒度。
# 功能说明：比较同一组标记在各分辨率的表达，确认粗/细列后保留原有各类绘图教学。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
coarse_key, fine_key = resolution_preview(ctx, adata, markers_2)
markers_1 = {k:v for k,v in markers_1.items() if v}
markers_2 = {k:v for k,v in markers_2.items() if v}


In [ ]:
# 功能说明：设置当前分析使用的聚类分辨率和标记基因集。
# 运行目的：为接下来的可视化（如 UMAP、点图）指定参数，针对低分辨率（0.02）进行大类注释。
# 详细代码解析：
# 1. `leiden_res = coarse_key`
#    - 将变量 `leiden_res` 设置为字符串 coarse_key，对应 `adata.obs` 中的列名。
# 2. `markers = markers_2`
#    - 将变量 `markers` 指向 `markers_2` 字典（大类标记基因）。

# 定义聚类分辨率变量
leiden_res = coarse_key
# 指定使用的markers版本
markers = markers_2

## 07.3 使用UMAP展示markergene

In [ ]:
# 功能说明：循环绘制不同细胞亚型的标记基因在 UMAP 上的表达分布。
# 运行目的：通过可视化已知标记基因的表达情况，来判断数据中哪些区域对应哪些细胞类型。
# 详细代码解析：
# 1. `for ct in markers:`
#    - 遍历之前定义的 B 细胞亚型列表。
# 2. `print(f"{ct.upper()}:")`
#    - 打印当前正在处理的细胞类型名称（转换为大写），作为标题。
# 3. `sc.pl.umap(...)`
#    - `sc.pl.umap`: Scanpy 的绘图函数，绘制 UMAP 图。
#    - `adata`: 数据对象。
#    - `color=markers_in_data[ct]`: 设置颜色依据。这里传入当前细胞类型对应的标记基因列表。Scanpy 会为列表中的每个基因生成一个子图，颜色深浅表示表达量。
#    - `vmin=0`: 设置颜色映射的最小值。
#    - `vmax="p99"`: 设置颜色映射的最大值。
#      - `"p99"`: 表示使用第 99 百分位数的表达值作为最大值。这可以防止极少数异常高表达的细胞（离群值）拉伸颜色范围，导致其他细胞看起来都是无色的。
#    - `sort_order=False`:
#      - `False`: 不对绘制顺序进行排序。默认情况下，Scanpy 可能会把表达量高的点画在上面。设置为 False 可以避免这种视觉偏差，更真实地反映平均表达情况。
#    - `frameon=False`: 不显示图形的边框。
#    - `cmap="Reds"`: 设置颜色映射表（Colormap）。"Reds" 表示从白色（低表达）到红色（高表达）的渐变。
# 4. `print("\n\n\n")`
#    - 打印三个换行符，增加输出之间的间距，使结果更易读。
with rc_context({"figure.figsize": (3, 3)}):
    for ct in markers:
        print(f"=== 可视化  {ct} 的标记基因 ===\n")  
        sc.pl.umap(
            adata,
            color = markers[ct]+ [leiden_res],
            vmin=0,
            vmax="p99", 
            sort_order=False, 
            frameon=False,
        #    cmap="Reds", 
            save=f"_07_178_{ct}.pdf"
            )
        print(f"=== 可视化  {ct} 的标记基因已完成 ===\n")  
        print("\n\n\n")  


## 07.4 使用小提琴图 (violin plot)展示markergene

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
for ct in markers:
    print(f"=== 可视化  {ct}的标记基因 ===\n")
    sc.pl.violin(
        adata,
        keys = markers[ct],
        groupby=leiden_res,
        stripplot=False,  # 移除内部的点
        inner="box",  # 在小提琴内部添加箱线图
        save=f"_07_180_{ct}.pdf",  # 使用多面板模式
    )


## 07.5 使用堆叠小提琴图 (stacked-violin plot)展示markergene

In [ ]:
# 功能说明：绘制堆叠小提琴图。
# 运行目的：紧凑地展示多个基因在多个聚类中的表达分布。
# 变量/函数/参数解析：
# - sc.pl.stacked_violin(...)：
#   - markers_dict：标记基因字典。
#   - groupby=fine_key：分组变量。
#   - swap_axes=False：不交换 x 轴和 y 轴。
#   - dendrogram=True：添加树状图。

sc.pl.stacked_violin(adata, markers, groupby=leiden_res, swap_axes=False, dendrogram=True,standard_scale='var',figsize=(18, 4),save="_07_182.pdf")

## 07.6 使用矩阵图 (matrixplot)展示markergene

In [ ]:
# 功能说明：绘制矩阵图（Matrix Plot）。
# 运行目的：以热图形式展示基因在各聚类中的平均表达量。
# 变量/函数/参数解析：
# - sc.pl.matrixplot(...)：
#   - cmap="Blues"：设置颜色映射为蓝色渐变。
#   - standard_scale="var"：
#     - 对每个变量（基因）进行标准化缩放，使其值在 0 到 1 之间。
#     - 这有助于比较不同基因在不同组间的相对表达模式，忽略绝对表达量的差异。
#   - colorbar_title="..."：设置颜色条的标题。
#    figsize，tuple[float, float] | None (默认值: None)，格式为 (宽度, 高度)
with rc_context({"figure.figsize": (18, 18)}):
    sc.pl.matrixplot(
        adata,
        markers,
        leiden_res,
        dendrogram=True,
        cmap="Blues",
        standard_scale="var",
        colorbar_title="column scaled\nexpression",
        figsize=(18, 4),
        save="_07_184.pdf"
    )

另一个有用的选项是使用 sc.pp.scale 对基因表达进行归一化。在这里，我们将此信息存储在 scale 层下。之后，我们调整绘图的最小值和最大值，并使用发散色图（在本例中为 RdBu_r，其中 _r 表示反转）。

In [ ]:
# 功能说明：对数据进行标准化（Z-score 归一化）并存储。
# 运行目的：计算每个基因的 Z-score（均值为 0，方差为 1），用于后续可视化，以便更清晰地展示高表达和低表达。
# 变量/函数/参数解析：
# - sc.pp.scale(adata, copy=True)：
#   - 对数据进行标准化。
#   - copy=True：返回一个新的 AnnData 对象，而不修改原对象。
# - .X：
#   - 获取标准化后的数据矩阵。
# - adata.layers["scaled"] = ...：
#   - 将标准化后的矩阵存储在 adata 的 layers 中，命名为 "scaled"。这样可以在不覆盖原始数据（.X）的情况下保存处理后的数据。

# 缩放并将结果存储在图层中
adata_scaled = scaled_view(adata, [g for genes in markers.values() for g in genes])


# 功能说明：使用标准化后的数据绘制矩阵图。
# 运行目的：展示基因表达的 Z-score，突出显示相对于平均水平的上调（红）或下调（蓝）。
# 变量/函数/参数解析：
# - layer="scaled"：
#   - 指定使用之前存储在 layers["scaled"] 中的数据进行绘图。
# - vmin=-2, vmax=2：
#   - 设置颜色映射的范围为 -2 到 2。超过此范围的值将被截断显示。这对于 Z-score 数据很常见。
# - cmap="RdBu_r"：
#   - 使用 "RdBu_r"（红-蓝反转）色图。通常红色表示高表达（正值），蓝色表示低表达（负值）。

sc.pl.matrixplot(
    adata_scaled,
    markers,
    leiden_res,
    dendrogram=True,
    colorbar_title="mean z-score",
    layer="scaled",
    vmin=-2,
    vmax=2,
    cmap="RdBu_r",
    figsize=(18, 4),
    save="_07_186.pdf"
)

## 07.7 使用热图 (Heatmaps)展示markergene

In [ ]:
# 功能说明：绘制单细胞分辨率的热图。
# 运行目的：展示每个细胞的基因表达情况，而不是分组平均值。
# 变量/函数/参数解析：
# - sc.pl.heatmap(...)：
#   - adata：AnnData 对象。
#   - markers_dict：标记基因。
#   - groupby="clusters"：按聚类排列细胞，并在图侧显示聚类颜色条。
#   - cmap="viridis"：颜色映射。
#   - dendrogram=True：显示聚类树状图。

sc.pl.heatmap(adata, markers, groupby=leiden_res, cmap="viridis", dendrogram=True,standard_scale="var", save="_07_188.pdf")

热图也可以绘制在缩放后的数据上，调整了最小值和最大值。

In [ ]:
# 功能说明：绘制基于标准化数据的热图，并交换轴。
# 运行目的：更清晰地展示基因表达模式，交换轴后基因在行，细胞在列（或反之，取决于默认）。
# 变量/函数/参数解析：
# - layer="scaled"：使用标准化数据。
# - vmin=-2, vmax=2：设置显示范围。
# - cmap="RdBu_r"：红蓝反转色图。
# - swap_axes=True：
#   - 交换 x 轴和 y 轴。默认情况下，热图的行是细胞，列是基因。交换后，行变成基因，列变成细胞。
# - figsize=(11, 4)：设置图形大小。

sc.pl.heatmap(
    adata_scaled,
    markers,
    groupby=leiden_res,
    layer="scaled",
    vmin=-2,
    vmax=2,
    cmap="RdBu_r",
    dendrogram=True,
  #  swap_axes=True,
   # figsize=(11, 4),
   save="_07_190.pdf",
)

## 07.8 使用点图 (dotplot)展示markergene

In [ ]:
# 功能说明：用点图展示各簇的标记基因表达水平。
# 运行目的：辅助为粗分辨率簇进行谱系级别的标签赋予。
# 变量/函数/参数解析：
# - sc.pl.dotplot(adata, markers, groupby=leiden_res, standard_scale="var")：
#   - groupby(str)：按哪一列分组显示，此处为低分辨率聚类标签。
#   - standard_scale(str)："var" 表示按基因维度标准化显示。
sc.pl.dotplot(adata, markers, groupby=leiden_res, standard_scale="var", save="_07_192.pdf")

In [ ]:
# 功能说明：重复绘制以便在不同分辨率下对比标签布局。
# 运行目的：辅助选择最终用于注释的分辨率。
with rc_context({"figure.figsize": (10, 8)}):
    sc.pl.umap(
        adata,
        color=[ leiden_res],
        legend_loc="on data",
    )

In [ ]:

# 使用 return_fig 参数返回图形对象
dot_plot = sc.pl.dotplot(
    adata, 
    markers, 
    groupby=leiden_res, 
    standard_scale="var",
    return_fig=True,
    show=False  # 不显示图形，只获取数据
)
# 检查可用的属性
print("可用的属性：", dir(dot_plot))
# 获取点图中的数据
if hasattr(dot_plot, 'dot_color_df'):
    print("颜色数据矩阵（标准化后的表达量）：")
    print(dot_plot.dot_color_df)
    
if hasattr(dot_plot, 'dot_size_df'):
    print("\n点大小数据矩阵（表达比例）：")
    print(dot_plot.dot_size_df)
    
# 最简单的方式：直接获取两个数据框
color_df = dot_plot.dot_color_df
size_df = dot_plot.dot_size_df
# 保存到CSV文件以便进一步分析
ctx.table("dotplot_color_coarse", color_df)
ctx.table("dotplot_size_coarse", size_df)

## 保存本章结果

保存表格、参数摘要和可供后续章节读取的数据。

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
adata.uns["course_markers"] = {"fine": markers_1, "broad": markers_2}
for label, key, group in [("coarse", coarse_key, markers_2), ("fine", fine_key, markers_1)]:
    dp = sc.pl.dotplot(adata, group, groupby=key, return_fig=True, show=False)
    ctx.table("marker_mean_" + label, dp.dot_color_df)
    ctx.table("marker_fraction_" + label, dp.dot_size_df)
ctx.finish(adata, {"marker_groups": list(markers_1), "missing_markers": missing_markers})


## 结果阅读与思考

请打开本次结果目录中的 summary.json、tables 和 figures。将目的、方法、结果和解释写入本章报告源稿，再更新 Word。

思考题：点图中点的大小和颜色分别代表什么？

运行与报告操作见课程根目录的 99_运行与AI协作指南.md。